In [0]:
from pyspark.sql.functions import current_timestamp
spark.sql('USE CATALOG delivery_cata')

DataFrame[]

In [0]:
spark.sql("""
INSERT OVERWRITE delivery_cata.gold_rpt.dim_driver
SELECT
    CAST(ROW_NUMBER() OVER (ORDER BY driver_id, effective_start) AS INT)  AS driver_sk,
    CAST(driver_id AS INT)          AS driver_id,
    name                            AS driver_name,
    phone                           AS driver_phone,
    status                          AS driver_status,
    effective_start,
    effective_end,
    is_current,
    current_timestamp()             AS _loaded_at
FROM delivery_cata.silver.drivers
""")
cnt = spark.table('delivery_cata.gold_rpt.dim_driver').count()
print(f'dim_driver: {cnt} rows ({spark.sql("SELECT COUNT(*) FROM delivery_cata.gold_rpt.dim_driver WHERE is_current").collect()[0][0]} current)')

dim_driver: 24175 rows (24175 current)


In [0]:
spark.sql("""
INSERT OVERWRITE delivery_cata.gold_rpt.dim_customer
SELECT
    CAST(ROW_NUMBER() OVER (ORDER BY user_id, effective_start) AS INT)    AS customer_sk,
    CAST(user_id AS INT)            AS customer_id,
    name                            AS customer_name,
    email                           AS customer_email,
    phone                           AS customer_phone,
    address                         AS customer_address,
    created_at                      AS customer_since,
    effective_start,
    effective_end,
    is_current,
    current_timestamp()             AS _loaded_at
FROM delivery_cata.silver.users
""")
cnt = spark.table('delivery_cata.gold_rpt.dim_customer').count()
print(f'dim_customer: {cnt} rows ({spark.sql("SELECT COUNT(*) FROM delivery_cata.gold_rpt.dim_customer WHERE is_current").collect()[0][0]} current)')

dim_customer: 23231 rows (23231 current)


In [0]:
spark.sql("""
INSERT OVERWRITE delivery_cata.gold_rpt.dim_vehicle
SELECT
    CAST(ROW_NUMBER() OVER (ORDER BY vehicle_id) AS INT)  AS vehicle_sk,
    CAST(vehicle_id AS INT)         AS vehicle_id,
    vehicle_type,
    plate_number,
    capacity_kg,
    status                          AS vehicle_status,
    current_timestamp()             AS _loaded_at
FROM delivery_cata.silver.vehicles
WHERE is_current = true
""")
cnt = spark.table('delivery_cata.gold_rpt.dim_vehicle').count()
print(f'dim_vehicle: {cnt} rows loaded')

dim_vehicle: 25926 rows loaded


In [0]:
spark.sql("""
INSERT OVERWRITE delivery_cata.gold_rpt.dim_date
WITH date_range AS (
    SELECT
        SEQUENCE(
            DATE(MIN(pickup_time)),
            DATE(MAX(COALESCE(delivery_time, pickup_time))),
            INTERVAL 1 DAY
        ) AS dates
    FROM delivery_cata.silver.deliveries
    WHERE is_current = true
),
exploded AS (
    SELECT EXPLODE(dates) AS full_date FROM date_range
)
SELECT
    CAST(DATE_FORMAT(full_date, 'yyyyMMdd') AS INT)     AS date_sk,
    full_date,
    YEAR(full_date)                                      AS year,
    QUARTER(full_date)                                   AS quarter,
    MONTH(full_date)                                     AS month,
    DATE_FORMAT(full_date, 'MMMM')                       AS month_name,
    WEEKOFYEAR(full_date)                                AS week_of_year,
    DAY(full_date)                                       AS day_of_month,
    DAYOFWEEK(full_date)                                 AS day_of_week,
    DATE_FORMAT(full_date, 'EEEE')                       AS day_name,
    DAYOFWEEK(full_date) IN (1, 7)                       AS is_weekend,
    current_timestamp()                                  AS _loaded_at
FROM exploded
ORDER BY full_date
""")
cnt = spark.table('delivery_cata.gold_rpt.dim_date').count()
print(f'dim_date: {cnt} dates loaded')

dim_date: 914 dates loaded


In [0]:
spark.sql("""
INSERT OVERWRITE delivery_cata.gold_rpt.dim_area
SELECT
    CAST(ROW_NUMBER() OVER (ORDER BY area_id) AS INT)     AS area_sk,
    CAST(area_id AS INT)            AS area_id,
    area_name,
    city,
    pincode,
    delivery_charge,
    current_timestamp()             AS _loaded_at
FROM delivery_cata.silver.areas
WHERE is_current = true
""")
cnt = spark.table('delivery_cata.gold_rpt.dim_area').count()
print(f'dim_area: {cnt} rows loaded')

dim_area: 24777 rows loaded


In [0]:
spark.sql("""
INSERT OVERWRITE delivery_cata.gold_rpt.dim_payment
SELECT
    CAST(ROW_NUMBER() OVER (ORDER BY payment_id) AS INT)  AS payment_sk,
    CAST(payment_id AS INT)         AS payment_id,
    payment_method,
    payment_status,
    payment_time,
    current_timestamp()             AS _loaded_at
FROM delivery_cata.silver.payments
WHERE is_current = true
""")
cnt = spark.table('delivery_cata.gold_rpt.dim_payment').count()
print(f'dim_payment: {cnt} rows loaded')

dim_payment: 7337 rows loaded


In [0]:

spark.sql("""
INSERT OVERWRITE delivery_cata.gold_rpt.fact_delivery
(
    delivery_id, order_id,
    driver_sk, customer_sk, vehicle_sk, area_sk, payment_sk, date_sk,
    order_status, delivery_status, pickup_location, delivery_location,
    order_amount, payment_amount, distance_km, delivery_time_mins, delivery_charge,
    pickup_time, delivery_time, order_date, payment_time,
    _loaded_at
)
SELECT
    CAST(d.delivery_id   AS INT)                                        AS delivery_id,
    CAST(d.order_id      AS INT)                                        AS order_id,

    dd.driver_sk,
    dc.customer_sk,
    dv.vehicle_sk,
    da.area_sk,
    dp.payment_sk,
    CAST(DATE_FORMAT(DATE(d.pickup_time), 'yyyyMMdd') AS INT)           AS date_sk,

    CASE WHEN o.order_status = 'SHIPPED'
         THEN 'IN_TRANSIT'
         ELSE o.order_status
    END                                                                 AS order_status,
    d.delivery_status,
    o.pickup_location,
    o.delivery_location,

    o.order_amount,
    p.amount                                                            AS payment_amount,
    d.distance_km,
    CAST(
        (unix_timestamp(d.delivery_time) - unix_timestamp(d.pickup_time))
        / 60 AS INT)                                                    AS delivery_time_mins,
    da.delivery_charge,

    d.pickup_time,
    d.delivery_time,
    o.created_at                                                        AS order_date,
    p.payment_time,
    current_timestamp()                                                 AS _loaded_at

FROM       delivery_cata.silver.deliveries       d

JOIN       delivery_cata.silver.orders           o
    ON  CAST(d.order_id  AS INT) = o.order_id
    AND o.is_current = true

-- is_current=true instead of SCD-2 time range
JOIN       delivery_cata.gold_rpt.dim_driver     dd
    ON  CAST(d.driver_id AS INT) = dd.driver_id
    AND dd.is_current = true

--is_current=true instead of SCD-2 time range  
JOIN       delivery_cata.gold_rpt.dim_customer   dc
    ON  CAST(o.user_id   AS INT) = dc.customer_id
    AND dc.is_current = true

LEFT JOIN  delivery_cata.gold_rpt.dim_vehicle    dv
    ON  CAST(d.vehicle_id AS INT) = dv.vehicle_id

-- area via driver's assigned area
LEFT JOIN  delivery_cata.silver.drivers          drv
    ON  CAST(d.driver_id  AS INT) = CAST(drv.driver_id AS INT)
    AND drv.is_current = true
LEFT JOIN  delivery_cata.gold_rpt.dim_area       da
    ON  CAST(drv.area_id  AS INT) = da.area_id

-- payment is optional
LEFT JOIN  delivery_cata.silver.payments         p
    ON  CAST(d.order_id   AS INT) = p.order_id
    AND p.is_current = true
LEFT JOIN  delivery_cata.gold_rpt.dim_payment    dp
    ON  CAST(p.payment_id AS INT) = dp.payment_id

WHERE d.is_current = true
""")

cnt = spark.table('delivery_cata.gold_rpt.fact_delivery').count()
print(f'fact_delivery: {cnt} rows')

fact_delivery: 8616 rows
